# Titanic Survival: Statistical Inference

This notebook asks whether key observed survival differences are compatible with chance under explicit null hypotheses. Tests are accompanied by effect sizes and confidence intervals. Statistical significance is not practical importance, and repeated testing requires caution.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy import stats
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
df=pd.read_csv(ROOT/'titanic.csv'); sns.set_theme(style='whitegrid', context='notebook')

## 1. Sex and survival

**Question.** Is the difference in survival proportions between women and men larger than expected under equal survival probabilities?

Method: Pearson chi-square test of independence, odds ratio, and a bootstrap 95% confidence interval for the risk difference. The observational design does not justify a causal interpretation.

In [ ]:
tab=pd.crosstab(df['Sex'],df['Survived']); print(tab)
chi2,p,dof,expected=stats.chi2_contingency(tab); print({'chi2':chi2,'dof':dof,'p_value':p})
female=df.loc[df.Sex=='female','Survived']; male=df.loc[df.Sex=='male','Survived']; diff=female.mean()-male.mean()
odds=(female.sum()/(len(female)-female.sum()))/(male.sum()/(len(male)-male.sum()))
rng=np.random.default_rng(42); boots=[]
for _ in range(5000): boots.append(rng.choice(female,len(female),replace=True).mean()-rng.choice(male,len(male),replace=True).mean())
print(f'Risk difference female-minus-male: {diff:.3f}; bootstrap 95% CI: {np.quantile(boots,[.025,.975])}; odds ratio: {odds:.2f}')

## 2. Passenger class and survival

**Question.** Does survival vary across the three passenger classes?

Method: chi-square test for a 3×2 table, followed by pairwise comparisons with Holm correction. The correction controls the family-wise error rate across the three class contrasts.

In [ ]:
tab=pd.crosstab(df.Pclass,df.Survived); print(tab)
print('Omnibus p-value:',stats.chi2_contingency(tab)[1])
pairs=[(1,2),(1,3),(2,3)]; ps=[]
for a,b in pairs:
    t=pd.crosstab(df.Pclass.isin([a]),df.Survived) if False else pd.crosstab(df.loc[df.Pclass.isin([a,b]),'Pclass'],df.loc[df.Pclass.isin([a,b]),'Survived'])
    ps.append(stats.chi2_contingency(t)[1])
print(pd.DataFrame({'contrast':[f'{a} vs {b}' for a,b in pairs],'raw_p':ps,'holm_p':stats.false_discovery_control(ps,method='bh') if False else np.minimum(1,np.array(ps)*np.arange(3,0,-1))}))

## 3. Continuous variables and distributional assumptions

**Question.** Do age and fare distributions differ between survivors and non-survivors?

Method: Mann–Whitney U tests because fare is strongly right-skewed and normality is not assumed; report rank-biserial effect sizes. A permutation test provides a distribution-free sensitivity check for the mean difference.

In [ ]:
for col in ['Age','Fare']:
    a=df.loc[df.Survived==1,col].dropna(); b=df.loc[df.Survived==0,col].dropna()
    u,p=stats.mannwhitneyu(a,b,alternative='two-sided'); r=1-2*u/(len(a)*len(b))
    print(f'{col}: survivor median={a.median():.2f}, non-survivor median={b.median():.2f}, U p={p:.4g}, rank-biserial r={r:.3f}')

## 4. Sensitivity and interpretation

The analyses support strong associations in this historical sample, but they do not resolve confounding: class, sex, age, family structure, and embarkation are not randomly assigned. P-values also do not measure the probability that a hypothesis is true. The next notebook uses held-out evaluation and interpretable models to quantify predictive rather than causal information.